# Network Routing Benchmark

In [1]:
%matplotlib inline
import time
import pandas as pd
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

from AnsatzBenchmarking.Builders.base import AnsatzBuilder
from AnsatzBenchmarking.Problems.networkTrafficRouting.NetworkTrafficRoutingProblems import NetworkTrafficRoutingProblemSet

from Utilities import cost_func, Estimator
from SLSQP import slsqp

from MomentumMonteCarlo import momentum_sa_phased as mb

In [2]:
class MonteCarloMomentumBuilder(AnsatzBuilder):
    def build(self):
        H = self.hamiltonian
        n_qubits = H.num_qubits
        
        circuit = QuantumCircuit(n_qubits)
        ansatz = QuantumCircuit(n_qubits)

        for i in range(n_qubits):
            pName = f"angle{i}"
            p = Parameter(pName)
            ansatz.rx(p, i)

        paramsList = [1] * n_qubits
        paramsIndex = [i for i in range(n_qubits)]

        # Call the Monte Carlo pipeline
        optimized_ansatz, optimized_params, _ = mb(
            params=paramsList,
            inds=paramsIndex,
            ansatz=ansatz,
            circuit=circuit,
            hamiltonian=H,
            estimator=StatevectorEstimator(),
            beta1=0.9,
            beta2=0.99,
            iters=3,                
            optimization_runs=500
        )

        self.circuit = optimized_ansatz
        self.optimized_params = np.array(optimized_params, dtype=float).tolist()

        return self.circuit

In [3]:
def wrap_cost(cost):
    """Ensure cost_func returns a scalar."""
    if isinstance(cost, np.ndarray):
        return float(cost.item() if cost.size == 1 else cost[0])
    if isinstance(cost, list):
        return float(cost[0] if len(cost) > 0 else 0)
    return float(cost)

def evaluate_builder_routing(builder_class, problem_set, trials=3):
    """Evaluate builder on Network Traffic Routing problems.

    Routing cost is a direct minimization
    """
    problems = problem_set.getProblemSet()
    results = []
    estimator = Estimator()

    for i, (h, exact) in enumerate(problems):
        print(f"\nProblem {i+1}/{len(problems)}: {h.num_qubits} qubits")
        hamiltonian = h  # minimize directly

        for trial in range(trials):
            try:
                builder = builder_class(hamiltonian)

                start = time.time()
                circuit = builder.getCircuit()
                build_time = time.time() - start

                start = time.time()
                params = np.random.random(len(circuit.parameters))
                opt_result = slsqp(
                    func=lambda p: wrap_cost(cost_func(p, circuit, hamiltonian, estimator)),
                    x0=params,
                    maxiter=200,
                )
                opt_time = time.time() - start

                min_cost_energy = opt_result.fun

                ansatz_name = builder_class.__name__.replace('Builder', '')
                if ansatz_name == 'FixedSU2':
                    ansatz_name = 'EfficientSU2'

                result = {
                    'ansatz_type': ansatz_name,
                    'problem_index': i + 1,
                    'trial': trial + 1,
                    'energy': min_cost_energy,
                    'exact_energy': exact if exact is not None else None,
                    'time': build_time + opt_time,
                    'params': len(circuit.parameters),
                    'converged': opt_result.success,
                }
                results.append(result)
                print(f"  Trial {trial+1}: ✓ Energy: {min_cost_energy:.6f}" +
                      (f" (exact: {exact:.6f})" if exact is not None else ""))
            except Exception as e:
                print(f"  Trial {trial+1} failed: {str(e)[:150]}")

    return results

In [4]:
# Load problem set and run benchmarks
print("Loading NetworkTrafficRouting problem set...")
problem_set = NetworkTrafficRoutingProblemSet()
problems = problem_set.getProblemSet()
print(f"Loaded {len(problems)} NetworkTrafficRouting problems\n")

print("Running benchmarks...")
print("="*60)

# Evaluate MomentumBuilder
print("\n" + "="*60)
print("MomentumBuilder:")
print("="*60)
momentum_results = evaluate_builder_routing(MonteCarloMomentumBuilder, problem_set, trials=3)

Loading NetworkTrafficRouting problem set...
Loaded 10 NetworkTrafficRouting problems

Running benchmarks...

MomentumBuilder:

Problem 1/10: 2 qubits
Energy after MomentumBuilder:  12.898689858616311
Energy after MomentumBuilder and Simulated Annealing (SA), two-phased:  5.002254599696078
  Trial 1: ✓ Energy: 5.000000 (exact: 5.000000)
Energy after MomentumBuilder:  12.898689858616311
Energy after MomentumBuilder and Simulated Annealing (SA), two-phased:  5.008609905311909
  Trial 2: ✓ Energy: 5.000000 (exact: 5.000000)
Energy after MomentumBuilder:  12.898689858616311
Energy after MomentumBuilder and Simulated Annealing (SA), two-phased:  5.011871256715176
  Trial 3: ✓ Energy: 5.000000 (exact: 5.000000)

Problem 2/10: 2 qubits
Energy after MomentumBuilder:  8.993587881883593
Energy after MomentumBuilder and Simulated Annealing (SA), two-phased:  3.0004763808775663
  Trial 1: ✓ Energy: 3.000000 (exact: 3.000000)
Energy after MomentumBuilder:  8.993587881883593
Energy after MomentumBui

KeyboardInterrupt: 